In [1]:
import pandas as pd 
import numpy as np
from datetime import datetime, date

In [2]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
    version="v2",
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)

https://api.allenneuraldynamics.org/v2/metadata_index/data_assets


In [3]:
aggregate = [
  {
    "$match": {
      "data_description.project_name": {
        "$regex": "Allen Brain Observatory - Visual Coding Neuropixels",
        "$options": "i"
      },
    }
  },
  {
    "$project": {
      "name": 1, 
      "subject_id": "$data_description.subject_id",
      "genotype": "$subject.subject_details.genotype", 
      "date_of_birth": "$subject.subject_details.date_of_birth", 
      "sex": "$subject.subject_details.sex", 
      "session_time": "$acquisition.acquisition_start_time",
      "project_name": "$data_description.project_name", 
      "modality": "$data_description.modalities.name",
      "session_type": "$acquisition.acquisition_type",
    }
  },
]
    
records = docdb_api_client.aggregate_docdb_records(
    pipeline = aggregate,
)

In [4]:
df = pd.DataFrame(records)

df['session_date'] = df.apply(lambda x: datetime.fromisoformat(x['session_time']).date(), axis=1)
df['session_time'] = df.apply(lambda x: datetime.fromisoformat(x['session_time']).time(), axis=1)
df['date_of_birth'] = df.apply(lambda x: datetime.strptime(x['date_of_birth'], '%Y-%m-%d').date(), axis=1)
df['age'] = df.apply(lambda x: (x['session_date'] - x['date_of_birth']).days, axis=1)

order = ['project_name','_id','name','subject_id','genotype','date_of_birth','sex','modality',
         'session_type','session_date','age','session_time']
df = df[order]

df.head()

,project_name,_id,name,subject_id,genotype,date_of_birth,sex,modality,session_type,session_date,age,session_time
0,Allen Brain Observatory - Visual Coding Neurop...,28a94fad-584c-42d3-8dea-8c2762d67075,433891_2019-02-27_13-09-23_nwb_2026-08-19_09-5...,433891,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,2018-11-08,Male,"[Behavior videos, Extracellular electrophysiol...",functional_connectivity,2019-02-27,111,13:09:23
1,Allen Brain Observatory - Visual Coding Neurop...,e580d6e8-7b56-4c62-befd-b35f66923fb4,405755_2018-09-12_13-32-59_nwb_2026-08-19_08-1...,405755,Vip-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,2018-06-12,Female,"[Behavior videos, Extracellular electrophysiol...",brain_observatory_1.1,2018-09-12,92,13:32:59
2,Allen Brain Observatory - Visual Coding Neurop...,f00b7e0f-4d0b-4c7d-89d5-aa4d63d98959,424448_2018-12-20_13-43-28_nwb_2026-08-19_09-4...,424448,wt/wt,2018-08-14,Male,"[Behavior videos, Extracellular electrophysiol...",brain_observatory_1.1,2018-12-20,128,13:43:28
3,Allen Brain Observatory - Visual Coding Neurop...,3a85289f-f7d6-4107-a51b-79767ca97a1a,386129_2018-06-27_14-07-11_nwb_2026-08-19_07-4...,386129,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,2018-03-02,Male,"[Behavior videos, Extracellular electrophysiol...",brain_observatory_1.1,2018-06-27,117,14:07:11
4,Allen Brain Observatory - Visual Coding Neurop...,1a251996-1a48-45f4-a5cb-27da1c6909f1,437660_2019-03-20_14-36-29_nwb_2026-08-19_10-0...,437660,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,2018-11-26,Male,"[Behavior videos, Extracellular electrophysiol...",functional_connectivity,2019-03-20,114,14:36:29


In [6]:
df.to_csv('/data/metadata/visual_coding_neuropixels_metadata.csv', index= False)